# ResuMatch — Exploration Notebook
EDA, model selection experiments, and scoring analysis.

In [2]:
!pip install matplotlib

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.1 MB 4.2 MB/s eta 0:00:02
   --------- ------------------------------ 1.8/8.1 MB 4.2 MB/s eta 0:00:02
   --------------- ------------------------ 3.1/8.1 MB 4.7 MB/s eta 0:00:02
   ----------------------------- ---------- 6.0/8.1 MB 5.7 MB/s eta 0:00:01
   ---------------------------------------- 8.1/8.1 MB 6.3 MB/s  0:00:01
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---------------------------------------- 2.3/2.3 MB 12.2 MB/s  0:00:00

   ---------------------------------------- 0/6 [pyparsing]
   ---------------------------------------- 0/6 [pyparsing]
   ---------------------------------------- 0/6 [pyparsing]
   ---------------------------------------- 0/6 [pyparsing]
   ------ --------------------------------- 1/6 [kiwisolver]
   ------ --------------------------

In [4]:
!pip install seaborn

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [5]:
import sys
sys.path.insert(0, '..')

from src.scorer import score
from src.parser import parse_sections
from src.keywords import keyword_gap_analysis
from src.similarity import compute_similarity
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
print('Imports OK')

Imports OK


## 1. Load Sample Data

In [6]:
with open('../data/samples/sample_resume.txt') as f:
    resume = f.read()

with open('../data/samples/sample_jd.txt') as f:
    jd = f.read()

print('Resume length:', len(resume.split()), 'words')
print('JD length:', len(jd.split()), 'words')

Resume length: 253 words
JD length: 230 words


## 2. Run Full Scoring Pipeline

In [9]:
result = score(resume, jd)

print(f"Overall Score     : {result['overall_score']}%")
print(f"Semantic Score    : {round(result['semantic_score']*100, 1)}%")
print(f"Keyword Match Rate: {round(result['keyword_gap']['match_rate']*100, 1)}%")
print(f"\nSection Scores:")
for k, v in result['section_scores'].items():
    print(f"  {k.capitalize():12}: {round(v*100,1) if v else 'N/A'}%")

TypeError: score() missing 1 required positional argument: 'model'

## 3. Keyword Gap Visualization

In [ ]:
gap = result['keyword_gap']
print('Matched keywords:', gap['matched'])
print('Missing keywords:', gap['missing'])
print('Extra keywords:  ', gap['extra'][:10])

# Bar chart
fig, ax = plt.subplots(figsize=(6, 3))
counts = [len(gap['matched']), len(gap['missing']), len(gap['extra'])]
labels = ['Matched', 'Missing', 'Extra']
colors = ['#6366f1', '#fca5a5', '#a5b4fc']
ax.bar(labels, counts, color=colors, width=0.5)
ax.set_title('Keyword Gap Summary', fontsize=13)
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## 4. Model Comparison: SBERT vs TF-IDF Cosine

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# TF-IDF cosine
vec = TfidfVectorizer(stop_words='english')
tfidf_matrix = vec.fit_transform([resume, jd])
tfidf_score = cosine_similarity(tfidf_matrix[0], tfidf_matrix[1])[0][0]

# SBERT
sbert_score = result['semantic_score']

print(f"TF-IDF Cosine Similarity : {round(tfidf_score*100, 1)}%")
print(f"SBERT Cosine Similarity  : {round(sbert_score*100, 1)}%")
print()
print('Insight: SBERT captures semantic meaning even when exact words differ.')
print('TF-IDF is purely lexical — misses synonyms and paraphrases.')

# Visual comparison
fig, ax = plt.subplots(figsize=(5, 3))
ax.barh(['TF-IDF', 'SBERT'], [tfidf_score*100, sbert_score*100],
        color=['#e0e7ff', '#6366f1'], height=0.4)
ax.set_xlabel('Similarity (%)')
ax.set_xlim(0, 100)
ax.set_title('TF-IDF vs SBERT Similarity')
for i, v in enumerate([tfidf_score*100, sbert_score*100]):
    ax.text(v + 1, i, f'{v:.1f}%', va='center')
plt.tight_layout()
plt.show()

## 5. Section-wise Score Heatmap

In [ ]:
sec_scores = {k: round(v*100,1) for k, v in result['section_scores'].items() if v}
df = pd.DataFrame([sec_scores], index=['Resume vs JD'])

plt.figure(figsize=(6, 1.5))
sns.heatmap(df, annot=True, fmt='.1f', cmap='RdYlGn',
            vmin=0, vmax=100, linewidths=0.5, cbar_kws={'label': 'Score'})
plt.title('Section-wise Similarity Scores (%)')
plt.tight_layout()
plt.show()